# einsum-contraction — worked example 1: Batched matrix multiply as an einsum

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `einsum-contraction`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

An einsum pattern names every axis; an index that appears on both inputs but is missing from the output is *contracted* (summed over), while an index on every operand and on the output is a passthrough batch axis. Batched matmul is the canonical case: the batch index is preserved and the shared inner dimension is contracted.

## Worked solution

We want `(B, I, K) = A @ B` for stacked matrices `A: (B, I, J)` and `Bm: (B, J, K)`.

1. Identify the roles of each index. `b` appears on both inputs and on the output, so it is a batch axis carried straight through. `i` comes from the first operand only and survives to the output. `k` comes from the second operand only and survives. `j` appears on both inputs but NOT on the output — that is the index we contract (sum over).
2. Write the pattern `'b i j, b j k -> b i k'`. The comma separates operands; the `->` separates inputs from output.
3. `einops.einsum` reads that string and, for each `(b, i, k)`, computes `sum_j A[b,i,j] * Bm[b,j,k]` — exactly the matmul definition.
4. The result is `(B, I, K)`, identical to `A @ Bm`, which we assert to confirm the index bookkeeping is right.

In [ ]:
import torch as t
import einops

t.manual_seed(0)
A = t.randn(2, 3, 4)
Bm = t.randn(2, 4, 5)

def batched_matmul(A, Bm):
    return einops.einsum(A, Bm, 'b i j, b j k -> b i k')

out = batched_matmul(A, Bm)
print(out.shape)
print('matches @:', bool(t.allclose(out, A @ Bm, atol=1e-5)))